# Import

In [1]:
from src.utils_worldmove import process_region, load_trajectory_lines
from src.utils_edit_graph import (handle_click,create_bridge)
from src.utils_elevation import download_opentopography_dem
from src.utils_computecost import bikespeed_from_power
from src.utils_mapgeometry import get_edge_geometry, route_nodes_to_geometry

import osmnx as ox
import networkx as nx
import geopandas as gpd
import math
from tqdm.auto import tqdm



from ipyleaflet import (
    Map,
    GeoJSON,
    Marker,
    Polyline,
    basemaps
)


# Falls nötig vorhher installieren

import geopandas as gpd
import osmnx as ox
import pyproj
import json

from shapely.geometry import LineString


print(f'OSMnx Verion: {ox.__version__}')
print(f'Networkx Verion: {nx.__version__}')

OSMnx Verion: 2.1.0
Networkx Verion: 3.6.1


# 1 Daten einlesen
[https://fi.ee.tsinghua.edu.cn/worldmove/](https://fi.ee.tsinghua.edu.cn/worldmove/)


In [2]:
# Process Administrative Boundaries
region, polygon, polygon_buffered = process_region(
    "data/Admin_Boundaries/regions.shp",  
    "output/City_Polygon.json", 
    target_crs="EPSG:3857",
    buffer_distance=1000)

# polygon.explore()

# -----------------------------------------------------------

# Process Trajectories

gdf_points, gdf_lines = load_trajectory_lines(
    "data/trajectories.npz",
    polygon=polygon,
    randomise=True,
    crs_output=3857
)
# gdf_points.head(3000).explore()

Regions in crs EPSG:32632
Prozess abgeschlossen.
[<POINT (8.509 47.39)> <POINT (8.509 47.39)> <POINT (8.516 47.402)> ...
 <POINT (8.582 47.306)> <POINT (8.582 47.306)> <POINT (8.582 47.306)>]


## Menge der Reisen Reduzieren

In [3]:
samplesize = 3000 
gdf_lines = gdf_lines.sample(n=samplesize)



# 2 Graph herunterladen

In [4]:
gdf_peri = polygon.to_crs(epsg=4326)
bbox = gdf_peri.union_all().envelope
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs=gdf_peri.crs)
bbox = bbox_gdf.geometry.iloc[0]

G_4326 = ox.graph.graph_from_polygon(bbox, network_type="drive", simplify=True)
G = G_4326
G_3857 = ox.project_graph(G_4326)


# 3 Graph bearbeiten

In [19]:
# als gpdf speichern (pro EPSG)
nodes_4326, edges_4326 = ox.graph_to_gdfs(G_4326)
nodes_3857, edges_3857 = ox.graph_to_gdfs(G_3857)

# leere Listen für def handle click und create bridge definieren
clicked_lines = []
clicked_points = []
clicked_markers = []
created_edges = []

# Karte Darstellen mit Jupyter Leaflet (Leaflet in Lat/Lon, 4326 ! Routing in WebMerc 3857 da Metrisch!)
# nur mal Grunddefinition, wird unten mit Funktionen ergänzt

center_lat = nodes_4326.geometry.y.mean()
center_lon = nodes_4326.geometry.x.mean()

m = Map(
    center=(center_lat, center_lon),
    zoom=13,
    basemap=basemaps.OpenStreetMap.Mapnik
)

edges_json = json.loads(edges_4326.to_json())
nodes_json = json.loads(nodes_4326.to_json())

hover_style = {
    "color": "yellow",
    "fillColor": "yellow",
    "fillOpacity": 1.0
}

edge_layer = GeoJSON(
    data=edges_json,
    style={
        "color": "gray",
        "weight": 1
    }
)

node_layer = GeoJSON(
    data=nodes_json,
    point_style={
        "radius": 3,
        "color": "blue",
        "fillColor": "blue",
        "fillOpacity": 0.8,
        "weight": 1
    },
    hover_style=hover_style
)


m.add_layer(edge_layer)
m.add_layer(node_layer)

#m      # aktivieren um Karte bereits hier anzuzeigen

In [20]:
def create_bridge(click1, click2):

    # Klicks von WGS84 -> projiziertes CRS (WebMerc 3857) transformieren

    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326",
        G_3857.graph["crs"],
        always_xy=True
    )

    x1, y1 = transformer.transform(click1[1], click1[0])
    x2, y2 = transformer.transform(click2[1], click2[0])

    # Nearest Nodes im projizierten Graphen

    node1 = ox.distance.nearest_nodes(
        G_3857,
        x1,
        y1
    )

    node2 = ox.distance.nearest_nodes(
        G_3857,
        x2,
        y2
    )

    # Koordinaten holen (projiziert -> Meter) für Längenberechung (Kosten)

    p1_3857 = (
        G_3857.nodes[node1]["x"],
        G_3857.nodes[node1]["y"]
    )

    p2_3857 = (
        G_3857.nodes[node2]["x"],
        G_3857.nodes[node2]["y"]
    )

    # Länge korrekt berechnen

    line_3857 = LineString([p1_3857, p2_3857])

    length = line_3857.length

    # Zurücktransformieren nach WGS84
    #    (für Darstellung auf Leaflet)

    reverse_transformer = pyproj.Transformer.from_crs(
        G_3857.graph["crs"],
        "EPSG:4326",
        always_xy=True
    )

    lon1, lat1 = reverse_transformer.transform(*p1_3857)
    lon2, lat2 = reverse_transformer.transform(*p2_3857)

    p1_4326 = (lon1, lat1)
    p2_4326 = (lon2, lat2)

    line_4326 = LineString([p1_4326, p2_4326])

    # Edge im projizierten Graphen hinzufügen

    G_3857.add_edge(
        node1,
        node2,
        length=float(length),
        geometry=line_3857,
        custom_edge=True
    )

    created_edges.append((G_3857, node1, node2))

    G_3857.add_edge(
        node2,
        node1,
        length=float(length),
        geometry=LineString([p2_3857, p1_3857]),
        custom_edge=True
    )

    created_edges.append((G_3857, node2, node1))


    # Edge zusätzlich im 4326 Graphen
    #    für Kartenanzeige

    G_4326.add_edge(
        node1,
        node2,
        length=float(length),
        geometry=line_4326,
        custom_edge=True
    )

    created_edges.append((G_4326, node1, node2))

    G_4326.add_edge(
        node2,
        node1,
        length=float(length),
        geometry=LineString([p2_4326, p1_4326]),
        custom_edge=True
    )

    created_edges.append((G_4326, node2, node1))

    # Linie auf Karte anzeigen  (! Leaflet erwartet Lat, Lon -> gpdf arbeitet mit Lon, Lat!)

    bridge_line = Polyline(
        locations=[
        (lat1, lon1),
        (lat2, lon2)
        ],
        color="red",
        weight=5
    )

    clicked_lines.append(bridge_line)

    m.add_layer(bridge_line)
    

In [21]:
def handle_click(**kwargs):     #kwargs = key-word-arguments

    # Nur echte Klicks verarbeiten
    if kwargs.get("type") != "click":
        return

    # Alte Marker/Linien/Edges entfernen
    if len(clicked_points) == 0:

        for marker in clicked_markers:
            m.remove_layer(marker)

        clicked_markers.clear()

        for line in clicked_lines:
            m.remove_layer(line)

        clicked_lines.clear()

        for graph, u, v in created_edges:

            if graph.has_edge(u, v):
                graph.remove_edge(u, v)

        created_edges.clear()

    # Klickkoordinaten
    lat, lon = kwargs.get("coordinates")

    # Punkt speichern, Marker sezten
    clicked_points.append((lat, lon))

    marker = Marker(location=(lat, lon))

    clicked_markers.append(marker)

    m.add_layer(marker)

    # Sobald zwei Punkte vorhanden, funktion von oben aufrufen
    if len(clicked_points) == 2:

        create_bridge(
            clicked_points[0],
            clicked_points[1]
        )

        # Punkte zurücksetzen
        clicked_points.clear()
    

In [22]:
# click Handler auf Interaktion (click) zu Karte hinzufügen
m.on_interaction(handle_click)


# karte Anzeigen und Funktionen Ausführen (Brücken Bauen ;)
# Jeweils eine Brücke aufs Mal, auch mehrer möglich -> Funktion handle click anpassen

m

Map(center=[np.float64(47.36803176581384), np.float64(8.50903755630625)], controls=(ZoomControl(options=['posi…

In [23]:
G_edit = G_4326
nodes_2, edges_2 = ox.convert.graph_to_gdfs(G_edit)

print(edges_2.shape)

edges_2.head()

(36052, 17)


osmid  \
u      v          key                                              
453735 589028154  0                                      4072320   
       92280288   0              [438517756, 4310693, 383751583]   
453740 453751     0    [290524400, 5880347, 210309180, 38578991]   
453751 4299948087 0              [383751584, 183852057, 5880347]   
       536792260  0                                     42879496   

                             highway   lanes    maxspeed oneway    ref  \
u      v          key                                                    
453735 589028154  0    motorway_link       3          80   True  A1;A3   
       92280288   0    motorway_link       2         100   True    A1H   
453740 453751     0         motorway  [3, 4]  [100, 120]   True    A1H   
453751 4299948087 0         motorway       3         120   True    A1H   
       536792260  0    motorway_link     NaN         NaN   True    NaN   

                      reversed       length  \
u      v          key                         
453735 589028154  0      False   279.196035   
       92280288   0      False   798.008882   
453740 453751     0      False  1647.000717   
453751 4299948087 0      False   706.064467   
       536792260  0      False   120.592096   

                                                                geometry  \
u      v          key                                                      
453735 589028154  0    LINESTRING (8.42293 47.41375, 8.42324 47.41357...   
       92280288   0    LINESTRING (8.42293 47.41375, 8.4233 47.41364,...   
453740 453751     0    LINESTRING (8.43409 47.41045, 8.43417 47.41044...   
453751 4299948087 0    LINESTRING (8.4549 47.40784, 8.45555 47.40741,...   
       536792260  0    LINESTRING (8.4549 47.40784, 8.45546 47.40738,...   

                      bridge name tunnel width junction access custom_edge  \
u      v          key                                                        
453735 589028154  0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
       92280288   0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
453740 453751     0      yes  NaN    NaN   NaN      NaN    NaN         NaN   
453751 4299948087 0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
       536792260  0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   

                      est_width  
u      v          key            
453735 589028154  0         NaN  
       92280288   0         NaN  
453740 453751     0         NaN  
453751 4299948087 0         NaN  
       536792260  0         NaN

In [24]:
nodes, edges = ox.convert.graph_to_gdfs(G)

print(edges.shape)
edges.head()

(36052, 17)


osmid  \
u      v          key                                              
453735 589028154  0                                      4072320   
       92280288   0              [438517756, 4310693, 383751583]   
453740 453751     0    [290524400, 5880347, 210309180, 38578991]   
453751 4299948087 0              [383751584, 183852057, 5880347]   
       536792260  0                                     42879496   

                             highway   lanes    maxspeed oneway    ref  \
u      v          key                                                    
453735 589028154  0    motorway_link       3          80   True  A1;A3   
       92280288   0    motorway_link       2         100   True    A1H   
453740 453751     0         motorway  [3, 4]  [100, 120]   True    A1H   
453751 4299948087 0         motorway       3         120   True    A1H   
       536792260  0    motorway_link     NaN         NaN   True    NaN   

                      reversed       length  \
u      v          key                         
453735 589028154  0      False   279.196035   
       92280288   0      False   798.008882   
453740 453751     0      False  1647.000717   
453751 4299948087 0      False   706.064467   
       536792260  0      False   120.592096   

                                                                geometry  \
u      v          key                                                      
453735 589028154  0    LINESTRING (8.42293 47.41375, 8.42324 47.41357...   
       92280288   0    LINESTRING (8.42293 47.41375, 8.4233 47.41364,...   
453740 453751     0    LINESTRING (8.43409 47.41045, 8.43417 47.41044...   
453751 4299948087 0    LINESTRING (8.4549 47.40784, 8.45555 47.40741,...   
       536792260  0    LINESTRING (8.4549 47.40784, 8.45546 47.40738,...   

                      bridge name tunnel width junction access custom_edge  \
u      v          key                                                        
453735 589028154  0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
       92280288   0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
453740 453751     0      yes  NaN    NaN   NaN      NaN    NaN         NaN   
453751 4299948087 0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   
       536792260  0      NaN  NaN    NaN   NaN      NaN    NaN         NaN   

                      est_width  
u      v          key            
453735 589028154  0         NaN  
       92280288   0         NaN  
453740 453751     0         NaN  
453751 4299948087 0         NaN  
       536792260  0         NaN

# Höhendaten hinzufügen

In [ ]:
# import os

# api_key = os.getenv("ELEVATION_API_KEY")

# path = download_opentopography_dem(
#     api_key=api_key,
#     output_file="data/global_copernicus_90m.tif",
#     demtype="COP90",
#     south=47.244256757894696,
#     north=47.47496700604382,
#     west=8.345124410649325,
#     east=8.637146966294877,
#     output_format="GTiff",
# )

# print(f"Gespeichert unter: {path}")

In [ ]:
G_edit = ox.elevation.add_node_elevations_raster(G_edit, 'data/global_copernicus_90m.tif', band=1, cpus=None)
G_edit = ox.elevation.add_edge_grades(G_edit, add_absolute=True)

G = ox.elevation.add_node_elevations_raster(G, 'data/global_copernicus_90m.tif', band=1, cpus=None)
G = ox.elevation.add_edge_grades(G, add_absolute=True)

# Kosten Vergeben

In [ ]:
G_edit = ox.projection.project_graph(G_edit, to_crs=3857, to_latlong=False)

nodes, edges = ox.convert.graph_to_gdfs(G_edit)


edges['bike_speed_mps'] = round(edges['grade'].apply(bikespeed_from_power),3)
edges['bike_traveltime'] = round(edges['length'] / edges['bike_speed_mps'],0)

nx.set_edge_attributes(G_edit, edges['bike_speed_mps'].to_dict(), 'bike_speed_mps')
nx.set_edge_attributes(G_edit, edges['bike_traveltime'].to_dict(), 'bike_traveltime')



In [ ]:

G = ox.projection.project_graph(G, to_crs=3857, to_latlong=False)

nodes, edges = ox.convert.graph_to_gdfs(G)


edges['bike_speed_mps'] = round(edges['grade'].apply(bikespeed_from_power),3)
edges['bike_traveltime'] = round(edges['length'] / edges['bike_speed_mps'],0)

nx.set_edge_attributes(G, edges['bike_speed_mps'].to_dict(), 'bike_speed_mps')
nx.set_edge_attributes(G, edges['bike_traveltime'].to_dict(), 'bike_traveltime')

# Start/Zielnodes vorbereiten

In [ ]:
gdf = gdf_lines

gdf["id"] = range(len(gdf))

gdf["start_x"] = gdf.geometry.apply(
    lambda line: line.coords[0][0]
)

gdf["start_y"] = gdf.geometry.apply(
    lambda line: line.coords[0][1]
)

gdf["end_x"] = gdf.geometry.apply(
    lambda line: line.coords[-1][0]
)

gdf["end_y"] = gdf.geometry.apply(
    lambda line: line.coords[-1][1]
)
gdf["source"] = ox.distance.nearest_nodes(
    G,
    X=gdf["start_x"].values,
    Y=gdf["start_y"].values
)

gdf["target"] = ox.distance.nearest_nodes(
    G,
    X=gdf["end_x"].values,
    Y=gdf["end_y"].values
)

gdf_edit = gpd.GeoDataFrame(gdf.copy(), geometry="geometry", crs=gdf.crs)
gdf_raw = gpd.GeoDataFrame(gdf.copy(), geometry="geometry", crs=gdf.crs)

# Heuristic Skale

In [ ]:
def heuristic_scale_from_bbox(bbox, safety_factor=0.99):
    if hasattr(bbox, "centroid"):
        center_lat = bbox.centroid.y
    else:
        minx, miny, maxx, maxy = bbox
        center_lat = (miny + maxy) / 2

    return math.cos(math.radians(abs(center_lat))) * safety_factor

heuristic_scale = heuristic_scale_from_bbox(bbox)

print(bbox)
print(f"Heuristik-Skalierung fuer A*: {heuristic_scale:.3f}")

# A* auf Zeit

In [ ]:
max_bike_speed_mps = edges["bike_speed_mps"].max()


def euclidean_heuristic_seconds(u, v):
    x1 = G.nodes[u]["x"]
    y1 = G.nodes[u]["y"]
    x2 = G.nodes[v]["x"]
    y2 = G.nodes[v]["y"]

    return math.hypot(x2 - x1, y2 - y1) * heuristic_scale / max_bike_speed_mps

In [ ]:
def compute_astar_time(G, source, target):
    try:
        route = nx.astar_path(
            G,
            source,
            target,
            heuristic=euclidean_heuristic_seconds,
            weight="bike_traveltime"
        )

        cost = sum(
            min(edge_data["bike_traveltime"] for edge_data in G[u][v].values())
            for u, v in zip(route[:-1], route[1:])
        )

        return route, cost

    except (
        nx.NetworkXNoPath,
        nx.NodeNotFound,
        KeyError
    ):
        return None, None


In [ ]:
route_pairs = list(zip(gdf["source"], gdf["target"]))
gdf_edit = gdf_edit.drop(columns=["route_nodes"], errors="ignore")

gdf_edit_routes = [
    compute_astar_time(G_edit, source, target)
    for source, target in tqdm(route_pairs, total=len(route_pairs))
]

gdf_edit["route_edit"] = [route for route, cost in gdf_edit_routes]
gdf_edit["cost_edit"] = [cost for route, cost in gdf_edit_routes]

gdf_edit["geometry"] = gdf_edit["route_edit"].apply(
    lambda route: route_nodes_to_geometry(G, route, weight="travel_time")
)

gdf_edit = gpd.GeoDataFrame(gdf_edit, geometry="geometry", crs=gdf.crs)


In [ ]:
route_pairs = list(zip(gdf["source"], gdf["target"]))

gdf_raw = gdf_raw.drop(columns=["route_nodes"], errors="ignore")

gdf_raw_routes = [
    compute_astar_time(G, source, target)
    for source, target in tqdm(route_pairs, total=len(route_pairs))
]

gdf_raw["route_raw"] = [route for route, cost in gdf_raw_routes]
gdf_raw["cost_raw"] = [cost for route, cost in gdf_raw_routes]

gdf_raw["geometry"] = gdf_raw["route_raw"].apply(
    lambda route: route_nodes_to_geometry(G, route, weight="travel_time")
)

gdf_raw = gpd.GeoDataFrame(gdf_raw, geometry="geometry", crs=gdf.crs)


In [ ]:
joined = gdf_edit.merge(
    gdf_raw.drop(columns="geometry"),
    on="id",
    how="left"
)

In [ ]:
joined['differenz'] = joined['cost_edit']-joined['cost_raw']

In [ ]:
joined.explore("differenz")